In [3]:
import numpy as np
import pandas as pd
import yfinance as yf

ticker_symbols = ["AAPL", "COST", "NVDA", "AMD", "WFC"]

start_date = "2020-01-01"
end_date = "2025-01-01" # alternatively can do None to set end date to today

prices = yf.download(ticker_symbols, start=start_date, end=end_date, progress=False, auto_adjust=False)["Adj Close"]
prices

Ticker,AAPL,AMD,COST,NVDA,WFC
Date,,,,,
2020-01-02,72.400505,49.099998,266.133972,5.971077,45.827175
2020-01-03,71.696640,48.599998,266.353088,5.875504,45.545799
2020-01-06,72.267937,48.389999,266.426117,5.900143,45.272976
2020-01-07,71.928040,48.250000,266.006134,5.971576,44.897831
2020-01-08,73.085106,47.830002,269.055542,5.982776,45.034252
...,...,...,...,...,...
2024-12-24,256.797211,126.290001,952.541382,140.181656,69.724731
2024-12-26,257.612701,125.059998,949.878967,139.891739,69.890312
2024-12-27,254.201370,125.190002,933.546509,136.972534,69.257240


In [4]:
import os
from dataclasses import dataclass
import pandas as pd
import numpy as np

In [5]:
pd.set_option("display.max_rows", 200)

@dataclass
class DataConfig:
    start: str = "2015-01-01"
    end: str | None = None
    interval: str = "1d"
    auto_adjust: bool = True

CFG = DataConfig()

# cell 2: fetch + normalize
def fetch_prices(tickers: list[str], cfg: DataConfig = CFG) -> pd.DataFrame:
    df = yf.download(
        tickers=tickers,
        start=cfg.start,
        end=cfg.end,
        interval=cfg.interval,
        auto_adjust=cfg.auto_adjust,
        progress=False,
        group_by="ticker"
    )

    # yfinance returns different shapes depending on tickers count
    if isinstance(df.columns, pd.MultiIndex):
        # take "Close" for each ticker
        closes = pd.DataFrame({t: df[t]["Close"] for t in tickers})
    else:
        closes = df[["Close"]].rename(columns={"Close": tickers[0]})

    closes = closes.dropna(how="all")
    closes.index = pd.to_datetime(closes.index)
    return closes

# cell 3: quick test
tickers = ["SPY", "AAPL", "MSFT", "NVDA"]
prices = fetch_prices(tickers)
prices.tail()

,SPY,AAPL,MSFT,NVDA
Date,,,,
2026-02-23,682.390015,266.179993,384.470001,191.550003
2026-02-24,687.349976,272.140015,389.000000,192.850006
2026-02-25,693.150024,274.230011,400.600006,195.559998
2026-02-26,689.299988,272.950012,401.720001,184.889999
2026-02-27,685.989990,264.179993,392.739990,177.190002
